In [1]:
!pip install -U langchain langchain-community langchain-core langchain-classic transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 69.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: requests


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate

import torch
import re
import json

In [3]:
model_name = "mistralai/Mistral-Nemo-Instruct-2407"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
def generate_text(prompt, max_new_tokens=600):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [5]:
full_name_schema = ResponseSchema(
    name="full_name",
    description="The candidate's full name as a string."
)

email_schema = ResponseSchema(
    name="email",
    description="The candidate's email address as a string."
)

education_schema = ResponseSchema(
    name="education",
    description=(
        "A list of education objects. Each object must contain "
        "degree, institution, and year."
    )
)

skills_schema = ResponseSchema(
    name="skills",
    description="A list of the candidate's skills as strings."
)

experience_schema = ResponseSchema(
    name="experience",
    description=(
        "A list of experience objects. Each object must contain "
        "role, company, and years."
    )
)

response_schemas = [
    full_name_schema,
    email_schema,
    education_schema,
    skills_schema,
    experience_schema
]

In [6]:
output_parser = StructuredOutputParser.from_response_schemas(
    response_schemas
)

format_instructions = output_parser.get_format_instructions()

print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The candidate's full name as a string.
	"email": string  // The candidate's email address as a string.
	"education": string  // A list of education objects. Each object must contain degree, institution, and year.
	"skills": string  // A list of the candidate's skills as strings.
	"experience": string  // A list of experience objects. Each object must contain role, company, and years.
}
```


In [7]:
user_input = """
John Smith – john.smith@email.com

Education:
B.Sc. Computer Science, MIT, 2020

Skills:
Python, Machine Learning, Data Analysis

Experience:
- Software Engineer at Google (2020–2023)
- Data Scientist at OpenAI (2023–Present)
"""

In [8]:
resume_extraction_template = """
You are an HR candidate profile parser.

Extract the candidate information from the resume text.

Return ONLY the extracted information in JSON format.

The required fields are:
- full_name
- email
- education
- skills
- experience

For education, each item must contain:
degree, institution, year.

For experience, each item must contain:
role, company, years.

Do not invent information.
If a field is not available, return an empty value or empty list as appropriate.

{format_instructions}

Resume text:
{user_input}
"""

In [9]:
prompt = PromptTemplate(
    template=resume_extraction_template,
    input_variables=[
        "user_input",
        "format_instructions"
    ]
).format(
    user_input=user_input,
    format_instructions=format_instructions
)

print(prompt)


You are an HR candidate profile parser.

Extract the candidate information from the resume text.

Return ONLY the extracted information in JSON format.

The required fields are:
- full_name
- email
- education
- skills
- experience

For education, each item must contain:
degree, institution, year.

For experience, each item must contain:
role, company, years.

Do not invent information.
If a field is not available, return an empty value or empty list as appropriate.

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The candidate's full name as a string.
	"email": string  // The candidate's email address as a string.
	"education": string  // A list of education objects. Each object must contain degree, institution, and year.
	"skills": string  // A list of the candidate's skills as strings.
	"experience": string  // A list of experience objects. Each object must co

In [10]:
response = generate_text(
    prompt,
    max_new_tokens=600
)

print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


```json
{
	"full_name": "John Smith",
	"email": "john.smith@email.com",
	"education": [
		{
			"degree": "B.Sc. Computer Science",
			"institution": "MIT",
			"year": "2020"
		}
	],
	"skills": [
		"Python",
		"Machine Learning",
		"Data Analysis"
	],
	"experience": [
		{
			"role": "Software Engineer",
			"company": "Google",
			"years": "2020–2023"
		},
		{
			"role": "Data Scientist",
			"company": "OpenAI",
			"years": "2023–Present"
		}
	]
}
```


In [11]:
def extract_json_block(text):
    fenced_pattern = r'```json\s*(.*?)\s*```'

    matches = re.findall(
        fenced_pattern,
        text,
        re.DOTALL | re.IGNORECASE
    )

    if matches:
        return matches[-1].strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        return text[start:end + 1].strip()

    raise ValueError("No JSON object found in model response.")

In [12]:
json_text = extract_json_block(response)

print(json_text)

{
	"full_name": "John Smith",
	"email": "john.smith@email.com",
	"education": [
		{
			"degree": "B.Sc. Computer Science",
			"institution": "MIT",
			"year": "2020"
		}
	],
	"skills": [
		"Python",
		"Machine Learning",
		"Data Analysis"
	],
	"experience": [
		{
			"role": "Software Engineer",
			"company": "Google",
			"years": "2020–2023"
		},
		{
			"role": "Data Scientist",
			"company": "OpenAI",
			"years": "2023–Present"
		}
	]
}


In [13]:
output_data = output_parser.parse(json_text)

print(output_data)

{'full_name': 'John Smith', 'email': 'john.smith@email.com', 'education': [{'degree': 'B.Sc. Computer Science', 'institution': 'MIT', 'year': '2020'}], 'skills': ['Python', 'Machine Learning', 'Data Analysis'], 'experience': [{'role': 'Software Engineer', 'company': 'Google', 'years': '2020–2023'}, {'role': 'Data Scientist', 'company': 'OpenAI', 'years': '2023–Present'}]}


In [14]:
print("Full Name:", output_data["full_name"])
print("Email:", output_data["email"])

print("\nEducation:")
for item in output_data["education"]:
    print(item)

print("\nSkills:")
for skill in output_data["skills"]:
    print(skill)

print("\nExperience:")
for item in output_data["experience"]:
    print(item)

Full Name: John Smith
Email: john.smith@email.com

Education:
{'degree': 'B.Sc. Computer Science', 'institution': 'MIT', 'year': '2020'}

Skills:
Python
Machine Learning
Data Analysis

Experience:
{'role': 'Software Engineer', 'company': 'Google', 'years': '2020–2023'}
{'role': 'Data Scientist', 'company': 'OpenAI', 'years': '2023–Present'}


In [15]:
assert "full_name" in output_data
assert "email" in output_data
assert "education" in output_data
assert "skills" in output_data
assert "experience" in output_data

assert isinstance(output_data["education"], list)
assert isinstance(output_data["skills"], list)
assert isinstance(output_data["experience"], list)

print("HR Candidate Profile Parser test passed successfully!")

HR Candidate Profile Parser test passed successfully!
